# Academic Risk Intelligence
## Explainable Decision Tree Classification — M.Tech IALM Engineering Project

**Student:** Abhishek Mishra  
**Roll Number:** 262220020  
**Programme:** M.Tech  
**Experiment:** Decision Tree Classification with Entropy, Information Gain and Model Optimization

### Abstract
This experiment develops an end-to-end classification workflow using a 1,000-row synthetic student dataset. The target is an illustrative binary label, `Academic_Risk`. The workflow covers data validation, exploratory data analysis, entropy and information gain, categorical preprocessing, model comparison, cross-validation, hyperparameter tuning, held-out evaluation, interpretability and prediction.

> **Dataset provenance and limitation:** The dataset is synthetic teaching data. Its target labels are generated from an illustrative rule with noise; they are not real student outcomes. Model metrics describe recovery of this synthetic rule and must not be interpreted as validated academic-risk performance or used to make consequential decisions about real students. Confirm that the instructor permits synthetic data and that the topic is unique among students.

## 1. Learning objectives
1. Explain classification, decision-tree structure, entropy and information gain.
2. Inspect and validate a structured dataset.
3. Prepare numeric and categorical features without leaking test information.
4. Train and compare entropy- and Gini-based decision trees.
5. Apply stratified cross-validation and GridSearchCV.
6. Evaluate the final model with class-sensitive metrics.
7. Interpret global feature importance, tree rules and an individual prediction path.

## 2. Theory: decision-tree classification

A decision tree is a supervised learning model that recursively partitions feature space into smaller regions. Each internal node tests a feature condition, each edge represents an outcome of that test, and each leaf assigns a class.

**Entropy** measures class impurity. For a node containing classes with proportions \(p_1,\ldots,p_k\):

\[
H(S)=-\sum_{i=1}^{k}p_i\log_2(p_i)
\]

A pure node has entropy 0. Entropy is largest when classes are evenly distributed.

**Information gain** measures the reduction in entropy after splitting on attribute \(A\):

\[
IG(S,A)=H(S)-\sum_{v\in Values(A)}\frac{|S_v|}{|S|}H(S_v)
\]

The tree chooses a split with high impurity reduction. `criterion="entropy"` uses entropy-based information gain; `criterion="gini"` uses Gini impurity. Scikit-learn's entropy criterion is an information-gain decision-tree implementation, not a claim that its complete implementation is identical to every historical ID3 variant.

**Overfitting control:** limiting `max_depth` and increasing `min_samples_leaf` constrain tree complexity. Cross-validation helps select settings that generalize better within the training data.

## 3. Import libraries and set reproducibility

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 4. Load dataset

In [ ]:
# Keep student_academic_risk_1000.csv in the same directory as this notebook.
DATA_PATH = Path("student_academic_risk_1000.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head(10))

## 5. Data dictionary and validation

In [ ]:
data_dictionary = pd.DataFrame({
    "Feature": df.columns,
    "Data type": [str(df[c].dtype) for c in df.columns],
    "Missing values": [int(df[c].isna().sum()) for c in df.columns],
    "Unique values": [int(df[c].nunique(dropna=True)) for c in df.columns]
})
display(data_dictionary)

print("Duplicate rows:", int(df.duplicated().sum()))
print("\\nTarget labels:")
display(df["Academic_Risk"].value_counts().rename_axis("Class").to_frame("Count"))

assert len(df) == 1000, "Expected exactly 1,000 records for this lab dataset."
assert "Academic_Risk" in df.columns, "Target column is missing."
assert df["Academic_Risk"].notna().all(), "Target contains missing values."

## 6. Exploratory data analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["Academic_Risk"].value_counts().plot(kind="bar", ax=axes[0], color=["#4472C4", "#ED7D31"])
axes[0].set_title("Target class distribution")
axes[0].set_xlabel("Academic risk"); axes[0].set_ylabel("Record count")
axes[0].tick_params(axis="x", rotation=0)

df["Program"].value_counts().plot(kind="bar", ax=axes[1], color="#70AD47")
axes[1].set_title("Records by academic programme")
axes[1].set_xlabel("Programme"); axes[1].set_ylabel("Record count")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_columns].hist(figsize=(14, 9), bins=22, edgecolor="white")
plt.suptitle("Numerical feature distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[numeric_columns].corr()
fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=75, ha="right")
ax.set_yticks(range(len(corr.index))); ax.set_yticklabels(corr.index)
ax.set_title("Correlation matrix — numerical predictors")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

### EDA interpretation guide
- Inspect class balance before choosing metrics; accuracy alone can hide poor minority-class detection.
- Histograms show the spread and shape of numeric variables.
- Correlation describes pairwise linear association between numeric features; it does not establish causation.
- The dataset is synthetic, so observed patterns reflect the generation process as well as random noise.

## 7. Define predictors and target

In [ ]:
TARGET = "Academic_Risk"
X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target classes:", sorted(y.unique()))

## 8. Training / testing design

A stratified 80/20 split reserves an independent test set while preserving approximate class proportions. All model selection and tuning are performed using the training partition only. The test partition is used for final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("\\nTraining class proportions:")
display(y_train.value_counts(normalize=True).rename("Proportion").to_frame())

## 9. Preprocessing pipeline

Categorical fields are one-hot encoded. Numeric fields pass through unchanged. The preprocessing transformer is placed inside a scikit-learn `Pipeline`, so it is fitted separately within each training fold during cross-validation. This avoids learning category mappings from held-out validation or test data.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

## 10. Baseline model: entropy criterion

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="entropy", max_depth=5, min_samples_leaf=10,
        random_state=RANDOM_STATE
    ))
])
baseline_model.fit(X_train, y_train)
print("Baseline entropy tree trained.")

## 11. Compare entropy and Gini criteria

In [ ]:
criterion_results = []
for criterion in ["entropy", "gini"]:
    candidate = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            criterion=criterion, max_depth=5, min_samples_leaf=10,
            random_state=RANDOM_STATE
        ))
    ])
    candidate.fit(X_train, y_train)
    candidate_pred = candidate.predict(X_test)
    criterion_results.append({
        "Criterion": criterion,
        "Accuracy": accuracy_score(y_test, candidate_pred),
        "Macro precision": precision_score(y_test, candidate_pred, average="macro", zero_division=0),
        "Macro recall": recall_score(y_test, candidate_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, candidate_pred, average="macro", zero_division=0)
    })

criterion_comparison = pd.DataFrame(criterion_results).set_index("Criterion")
display(criterion_comparison.round(4))
criterion_comparison[["Accuracy", "Macro F1"]].plot(kind="bar", figsize=(8, 4), ylim=(0, 1))
plt.title("Entropy vs Gini — held-out test comparison")
plt.ylabel("Score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 12. Stratified 5-fold cross-validation

Cross-validation estimates how consistently a model performs across multiple training/validation partitions. The test set remains untouched. Macro-averaged scores give each class equal weight.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(
    baseline_model, X_train, y_train, cv=cv,
    scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    return_train_score=True
)
cv_summary = pd.DataFrame({
    metric: cv_scores[f"test_{metric}"]
    for metric in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
})
display(cv_summary.agg(["mean", "std"]).T.rename(columns={"mean":"CV mean","std":"CV standard deviation"}).round(4))

## 13. Hyperparameter tuning using GridSearchCV

GridSearchCV evaluates parameter combinations using cross-validation on the training set. The chosen configuration maximizes macro F1, which balances precision and recall equally across classes. The search below is intentionally compact for a classroom dataset.

In [ ]:
parameter_grid = {
    "classifier__criterion": ["entropy", "gini"],
    "classifier__max_depth": [3, 4, 5, 6, None],
    "classifier__min_samples_split": [2, 10, 20],
    "classifier__min_samples_leaf": [5, 10, 20],
    "classifier__ccp_alpha": [0.0, 0.001, 0.005]
}

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    param_grid=parameter_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)
grid_search.fit(X_train, y_train)

print("Best cross-validation macro F1:", round(grid_search.best_score_, 4))
print("Best parameters:")
print(grid_search.best_params_)
best_model = grid_search.best_estimator_

In [ ]:
tuning_results = pd.DataFrame(grid_search.cv_results_)
display(
    tuning_results.sort_values("rank_test_score")[
        ["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]
    ].head(10)
)

## 14. Final held-out test evaluation

In [ ]:
y_pred = best_model.predict(X_test)
class_labels = list(best_model.named_steps["classifier"].classes_)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Macro precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0)
}
try:
    risk_index = class_labels.index("At Risk")
    metrics["ROC-AUC (At Risk)"] = roc_auc_score(
        (y_test == "At Risk").astype(int),
        best_model.predict_proba(X_test)[:, risk_index]
    )
except ValueError:
    metrics["ROC-AUC (At Risk)"] = np.nan

display(pd.Series(metrics, name="Test score").to_frame().round(4))
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_labels)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title("Final model — confusion matrix")
plt.tight_layout()
plt.show()

## 15. Visualize the optimized decision tree

In [ ]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
final_tree = best_model.named_steps["classifier"]
encoded_feature_names = fitted_preprocessor.get_feature_names_out()

fig, ax = plt.subplots(figsize=(22, 12))
plot_tree(
    final_tree, feature_names=encoded_feature_names,
    class_names=final_tree.classes_, filled=True, rounded=True,
    max_depth=4, fontsize=8, ax=ax
)
ax.set_title("Optimized Decision Tree — first four levels")
plt.tight_layout()
plt.show()

## 16. Explainability: feature importance and decision rules

In [ ]:
feature_importance = pd.Series(
    final_tree.feature_importances_, index=encoded_feature_names
).sort_values(ascending=False)

display(feature_importance.head(15).rename("Importance").to_frame())
feature_importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Top 15 encoded feature importances")
plt.xlabel("Mean decrease in impurity")
plt.tight_layout()
plt.show()

In [ ]:
print(export_text(final_tree, feature_names=list(encoded_feature_names), max_depth=5))

**Interpretation caution:** impurity-based importance can be biased toward features with many possible split points and can distribute importance across one-hot columns. It describes how this fitted tree uses the encoded inputs; it does not establish causal influence.

## 17. Local explanation: trace one prediction

In [ ]:
example_student = pd.DataFrame([{
    "Age": 20,
    "Attendance_Percent": 62.0,
    "Weekly_Study_Hours": 2.0,
    "Previous_Exam_Score": 48.0,
    "Assignments_Completed_Out_of_10": 4,
    "Average_Sleep_Hours": 6.0,
    "Internet_Access": "Average",
    "Class_Participation": "Low",
    "Program": "IT"
}])

example_prediction = best_model.predict(example_student)[0]
example_probabilities = best_model.predict_proba(example_student)[0]
print("Predicted class:", example_prediction)
print("Class probabilities:", dict(zip(final_tree.classes_, example_probabilities.round(4))))

In [ ]:
transformed_example = fitted_preprocessor.transform(example_student)
decision_path = final_tree.decision_path(transformed_example)
visited_nodes = decision_path.indices[
    decision_path.indptr[0]:decision_path.indptr[1]
]
tree_structure = final_tree.tree_

print("Nodes visited from root to leaf:")
for node_id in visited_nodes:
    feature_id = tree_structure.feature[node_id]
    if feature_id >= 0:
        feature_name = encoded_feature_names[feature_id]
        threshold = tree_structure.threshold[node_id]
        print(f"Node {node_id}: test {feature_name} <= {threshold:.4f}")
    else:
        print(f"Leaf node {node_id}: class counts = {tree_structure.value[node_id].ravel()}")

## 18. Manual entropy and information-gain demonstration

For a binary target with counts \(n_+\) and \(n_-\), first calculate:

\[
p_+=\frac{n_+}{n_++n_-},\quad p_-=\frac{n_-}{n_++n_-}
\]

\[
H(S)=-p_+\log_2(p_+)-p_-\log_2(p_-)
\]

For a candidate split into left and right subsets:

\[
H_{split}=\frac{N_L}{N}H(S_L)+\frac{N_R}{N}H(S_R)
\]

\[
IG=H(S)-H_{split}
\]

The next cell demonstrates this calculation for a median split on attendance using the training data.

In [ ]:
def calculate_entropy(labels):
    probabilities = pd.Series(labels).value_counts(normalize=True).to_numpy()
    return float(-np.sum(probabilities * np.log2(probabilities)))

base_entropy = calculate_entropy(y_train)
candidate_feature = "Attendance_Percent"
candidate_threshold = X_train[candidate_feature].median()
left = X_train[candidate_feature] <= candidate_threshold
right = ~left

left_entropy = calculate_entropy(y_train[left])
right_entropy = calculate_entropy(y_train[right])
weighted_split_entropy = (
    left.mean() * left_entropy + right.mean() * right_entropy
)
information_gain = base_entropy - weighted_split_entropy

print(f"Training-set entropy: {base_entropy:.4f} bits")
print(f"Candidate split: {candidate_feature} <= {candidate_threshold:.2f}")
print(f"Left: n={left.sum()}, entropy={left_entropy:.4f}")
print(f"Right: n={right.sum()}, entropy={right_entropy:.4f}")
print(f"Weighted post-split entropy: {weighted_split_entropy:.4f}")
print(f"Information gain: {information_gain:.4f} bits")

## 19. Engineering discussion and limitations

1. **Data provenance:** the dataset is synthetic, with labels generated from an illustrative rule plus noise.
2. **Generalization:** a held-out split and cross-validation provide internal estimates, but no external or real-world validation is performed.
3. **Class-sensitive evaluation:** precision, recall and F1 should be read alongside class distribution and confusion matrix.
4. **Model complexity:** depth and leaf-size constraints help manage overfitting; pruning is explored through `ccp_alpha`.
5. **Explainability:** decision paths show the conditions used by the tree, but do not prove causal relationships.
6. **Responsible use:** never use this demonstration to label, rank, penalize, or intervene with real students.

## 20. Conclusion

An end-to-end decision-tree classification workflow was implemented using a 1,000-record synthetic dataset. The experiment included data validation, exploratory analysis, entropy and information gain, categorical encoding, entropy/Gini comparison, stratified cross-validation, hyperparameter tuning, held-out evaluation, tree visualization, feature importance, decision rules and local prediction-path inspection.

The work demonstrates a reproducible machine-learning engineering process. Since the target labels are synthetic, the measured performance only reflects this generated dataset and is not evidence of real academic-risk prediction capability.

## Appendix A — Run the companion Streamlit dashboard

From the project directory, install dependencies and run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

The dashboard provides Overview, Data Quality & EDA, Model Performance, Explainability and Live Prediction tabs.